In [1]:
import os
import sys

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64/"
os.environ["SPARK_HOME"] = "/usr/local/spark/"
spark_path = os.environ['SPARK_HOME']
sys.path.append(spark_path + "/bin")
sys.path.append(spark_path + "/python")
sys.path.append(spark_path + "/python/pyspark/")
sys.path.append(spark_path + "/python/lib")
sys.path.append(spark_path + "/python/lib/pyspark.zip")
sys.path.append(spark_path + "/python/lib/py4j-0.10.9.7-src.zip")

number_cores = 8
memory_gb = 16

In [2]:
import findspark
import pyspark
import json

findspark.init()
conf = (pyspark.SparkConf().setMaster('local[{}]'.format(number_cores)).set('spark.driver.memory', '{}g'.format(memory_gb)))

sc = pyspark.SparkContext(conf=conf)

In [3]:
reviewFieldsRDD = sc.textFile("output/descAnalysis/reviewFieldsSample/part-00000").map(json.loads)
print("Sample Review Data:\n\nFormat: Review ID, Business ID, User ID, Stars, Review")
print(reviewFieldsRDD.take(1))

Sample Review Data:

Format: Review ID, Business ID, User ID, Stars, Review
[['KU_O5udG6zpxOg-VcAEodg', 'XQfwVwDr-v0ZS3_CbbE5Xw', 'mh_-eMZ6K5RLWhZyISBhwA', 3.0, "If you decide to eat here, just be aware it is going to take about 2 hours from beginning to end. We have tried it multiple times, because I want to like it! I have been to it's other locations in NJ and never had a bad experience. \n\nThe food is good, but it takes a very long time to come out. The waitstaff is very young, but usually pleasant. We have just had too many experiences where we spent way too long waiting. We usually opt for another diner or restaurant on the weekends, in order to be done quicker."]]


In [4]:
userFieldsRDD = sc.textFile("output/descAnalysis/userFieldsSample/part-00000").map(json.loads)
print("Sample User Data:\n\nFormat: User ID, Name, Account Creation Date, Friend(s) User ID")
userSample = userFieldsRDD.take(3)
print(userSample[2])

Sample User Data:

Format: User ID, Name, Account Creation Date, Friend(s) User ID
['2WnXYQFK0hXEoTxPtV2zvg', 'Steph', '2008-07-25 10:41:00', 'LuO3Bn4f3rlhyHIaNfTlnA, j9B4XdHUhDfTKVecyWQgyA, pypZb3V5TXHOnlTj-qLSrw, 7cDAEEnwfSqG2Lv8Vanr3Q, irkRHMqg9oSt7lv3OSiNkA, 6jEeCpNEU9l8CT9X566Oog, W5VmqP2T4O_aAMq5YwTJzQ, wzyY07YiGwgRKCIcsrBuRQ, IL2yTm7zGmsTF4iKVSZ0ug, AB9JedB0R5wWIsAGjXoMWw, 9AB3-Tw3DBfuEIr0HIXSpg, H0gxrggG__efXkGXRQP1_A, rNNFRAaaIpzJZ1KBrRorZw, FaYXjGgLtetpCCiKmEsq9g, ePSSiFQ8kvQ9-6nAa_pMBA, Tz1FRUGfq7xBf4-ZGlvLZg, wC5FDlL4d5dstl8q91_dQw, H5EY1CeIx4m_pNChKZkq7A, BIdFgt_4owlS1VUcTuZ4fA, 84FcN7Bq9BnL3wPtnyye6w, PCm1knzIbOaziikbVDz-jw, Sa48aWAqDy1wMRgRFGiRHw, DECwE-Th7VEa5PWWoru5iw, IVM7P6IjfWj_sUhLrL9FFg, gjauhhlrvelyFY2N63T8CA, iT1li5FhUudWmviBMMdS7w, 1scP-I_N0AsKLVdzPe3YIw, zHdltdICrilxbqdHFxtVBA, lYC-DFT7VOdxSM6oY2Vm1g, W-OZPeKh4R0GlywUsl09PA, 1unQbxCkgwrZMRmox79LCg, 5NfYjjb1E4c_StGEGXXUXQ, qNrHLZPurBWJzeAMkFLvvA, hmwKaQN_f0dnymp7XH2R2w, F_5_UNX-wrAFCXuAkBZRDw, fQq-PBKARvUGfBnmc

In [5]:
businessFieldsRDD = sc.textFile("output/descAnalysis/businessFieldsSample/part-00000").map(json.loads)
print("Sample Business Data:\n\nFormat: Business ID, Business Name, # of Reviews, Star Rating")
print(businessFieldsRDD.take(1))

Sample Business Data:

Format: Business ID, Business Name, # of Reviews, Star Rating
[['Pns2l4eNsfO8kk83dixA6A', 'Abby Rappoport, LAC, CMQ', 7, 5.0]]


In [6]:
lowRatedReviewsRDD = sc.textFile("output/descAnalysis/lowRatedReviewsSample").map(json.loads)
print("Low Rating Reviews:\n1-2 star reviews from users on a given business.\n\nFormat: Review ID, User ID, Business ID, Stars, Review")
print(lowRatedReviewsRDD.take(1))

Low Rating Reviews:
1-2 star reviews from users on a given business.

Format: Review ID, User ID, Business ID, Stars, Review
[['JrIxlS1TzJ-iCu79ul40cQ', '04UD14gamNjLY0IDYVhHJg', 'eUta8W_HdHMXPzLBBZhL1A', 1.0, "I am a long term frequent customer of this establishment. I just went in to order take out (3 apps) and was told they're too busy to do it. Really? The place is maybe half full at best. Does your dick reach your ass? Yes? Go fuck yourself! I'm a frequent customer AND great tipper. Glad that Kanella just opened. NEVER going back to dmitris!"]]


In [7]:
multiReviewOnBusinessRDD = sc.textFile("output/descAnalysis/finalResultSample/part-00000").map(json.loads)
print("Joined result of friends' reviews on the same business that also gave a low rating.\n\nFormat: Flagged User ID, [Business ID #1, Flagged User's Review ID, Friend Reviews [Friend User ID, Friend Review ID]], Business ID #2... so on")
print(multiReviewOnBusinessRDD.take(1))

Joined result of friends' reviews on the same business that also gave a low rating.

Format: Flagged User ID, [Business ID #1, Flagged User's Review ID, Friend Reviews [Friend User ID, Friend Review ID]], Business ID #2... so on
[{'userId': 'cRANsQ5E_sxeQ0Aoop3drg', 'businessReviews': [{'businessId': '5j9TfW8RrdkpaHlaBwt6bQ', 'userReviewId': 'Udlabx__W_s1UOjYNApIDg', 'friendsReviews': [['hw5zmHc2QaiO-LCIs5MqHA', 'wFwWCuRRfpwd_itwe5JNTA'], ['56oHQV84bjSMtLkIOsh_4A', 'VK6MoyM9WOkdeCS3jrcdxw'], ['bnKo6L4MgcUbLZH7fmnkrA', 'n3yCH2cjBrlgjbFDwkLCig'], ['bnKo6L4MgcUbLZH7fmnkrA', 'viRL3ynZum5V5YZMXMyQSg'], ['u0igpMz49lHLul-9yH9COw', 'usWDFj78csqY7T4lOgSG6A'], ['CqQuA6sT1_CmNJWR3qkzeg', '_V-jDh7J9OYQdZUVoGG-rA']]}, {'businessId': 'pAFtOcz710oLQ-e_J6AIzw', 'userReviewId': 'q5lvZuoGrdNZtgBY7AHJ0w', 'friendsReviews': [['YNkRUoruuV4g3jSDIyXw9A', 't66qj4cbPYAYlfpqxchYvg'], ['-vPtYuV_dMPb0Bbzg9ECVw', 'KIAqCeQuhC2BQNI-p9AXLw'], ['EmurkIplcpOkh4hUnSs_ww', 'HKege4lBV85T-flscfoXgQ']]}]}]


In [8]:
filteredReviews = sc.textFile("output/descAnalysis/finalResultSample/part-00000").map(json.loads)
print("Visualization of formatting for a flagged user:\n")
filteredReviews.take(1)

Visualization of formatting for a flagged user:



[{'userId': 'cRANsQ5E_sxeQ0Aoop3drg',
  'businessReviews': [{'businessId': '5j9TfW8RrdkpaHlaBwt6bQ',
    'userReviewId': 'Udlabx__W_s1UOjYNApIDg',
    'friendsReviews': [['hw5zmHc2QaiO-LCIs5MqHA', 'wFwWCuRRfpwd_itwe5JNTA'],
     ['56oHQV84bjSMtLkIOsh_4A', 'VK6MoyM9WOkdeCS3jrcdxw'],
     ['bnKo6L4MgcUbLZH7fmnkrA', 'n3yCH2cjBrlgjbFDwkLCig'],
     ['bnKo6L4MgcUbLZH7fmnkrA', 'viRL3ynZum5V5YZMXMyQSg'],
     ['u0igpMz49lHLul-9yH9COw', 'usWDFj78csqY7T4lOgSG6A'],
     ['CqQuA6sT1_CmNJWR3qkzeg', '_V-jDh7J9OYQdZUVoGG-rA']]},
   {'businessId': 'pAFtOcz710oLQ-e_J6AIzw',
    'userReviewId': 'q5lvZuoGrdNZtgBY7AHJ0w',
    'friendsReviews': [['YNkRUoruuV4g3jSDIyXw9A', 't66qj4cbPYAYlfpqxchYvg'],
     ['-vPtYuV_dMPb0Bbzg9ECVw', 'KIAqCeQuhC2BQNI-p9AXLw'],
     ['EmurkIplcpOkh4hUnSs_ww', 'HKege4lBV85T-flscfoXgQ']]}]}]

In [9]:
filteredUsers = sc.textFile("output/descAnalysis/filteredUsers/part-00000").map(json.loads)
print("Flagged user information from the yelp user dataset:")
filteredUsers.take(1)

Flagged user information from the yelp user dataset:


[{'user_id': 'cRANsQ5E_sxeQ0Aoop3drg',
  'name': 'Amanda',
  'review_count': 133,
  'yelping_since': '2013-11-27 00:27:57',
  'useful': 366,
  'funny': 80,
  'cool': 268,
  'elite': '2017,2018,2019,20,20,2021',
  'friends': 'Io4DrR1_vMLOVMdQsPATRA, LMIyGqwDMAVMWkCQb6sQ3Q, KlKgH4Te9EPb1uzC28Aw-w, ybN5igJTmnkFHXU4qseiDA, r12E0JbaVi4JCUgUa7b7fA, J2c6-nchh5ziQQuMOtm41A, ystMl1kYo0KMAnafP9hx6A, -SsCP5luPDPJyc1z0F4fHA, 3echCr8Y1XeW3Fq5Q_xn4w, vGDDsijFim4Zay8vlf6TDQ, kGd2F0sfE8WWAWty5uezNw, WHuuz1Xm8zkrUl8fAd31Eg, -ZTW3gcV_1DL1Eqx78DIAQ, du9gjqdmv6koj-kYpk3KQg, swobJ5rJuEGRKqHBf8EutA, 99HXweAV7ypgg0FNF0o8Jw, a5rkDUU_Ie2cDyktgtLyGQ, RaRJMKc-gXukTHFovDwiiw, EUkxMG8Bf6nGsrIXOCAQ4w, md5proR8D9ooUM_EHf7mdg, Vk_khMDkjXPkpffQXArFqA, 6oUE_bDkqfXap6zK5VWqIQ, nXgApfRajhkwCPPManEHPA, z4OYmpgfcaWMVIPGDKm_pg, XzYyKC84ccqgw0DccAgEsQ, q3_rELfkhP2JF8n78cBmRA, vwpYsmLp-g_VWbIt-vR3Qg, uM9OD-grlkb-MF3be1JQMw, T-ERRihmQAHhkIWVDTcSBA, uxlVKI_tSA-yMNDfffngJw, 5x3oDSoQWE3qqG_KEZA0oQ, 4jth6KJ1-EYVE9MHHhEkOA, qsGDRdJ

In [10]:
linkedReviews = sc.textFile("output/descAnalysis/linkedReviews/part-00000")
print("Associating a user trend with IDs can be difficult, this displays the user's name from their information above.\n")
print(linkedReviews.take(1))

Associating a user trend with IDs can be difficult, this displays the user's name from their information above.

["{'userId': 'cRANsQ5E_sxeQ0Aoop3drg', 'userName': 'Amanda', 'suspectReviewData': {'userId': 'cRANsQ5E_sxeQ0Aoop3drg', 'businessReviews': [{'businessId': '5j9TfW8RrdkpaHlaBwt6bQ', 'userReviewId': 'Udlabx__W_s1UOjYNApIDg', 'friendsReviews': [['hw5zmHc2QaiO-LCIs5MqHA', 'wFwWCuRRfpwd_itwe5JNTA'], ['56oHQV84bjSMtLkIOsh_4A', 'VK6MoyM9WOkdeCS3jrcdxw'], ['bnKo6L4MgcUbLZH7fmnkrA', 'n3yCH2cjBrlgjbFDwkLCig'], ['bnKo6L4MgcUbLZH7fmnkrA', 'viRL3ynZum5V5YZMXMyQSg'], ['u0igpMz49lHLul-9yH9COw', 'usWDFj78csqY7T4lOgSG6A'], ['CqQuA6sT1_CmNJWR3qkzeg', '_V-jDh7J9OYQdZUVoGG-rA']]}, {'businessId': 'pAFtOcz710oLQ-e_J6AIzw', 'userReviewId': 'q5lvZuoGrdNZtgBY7AHJ0w', 'friendsReviews': [['YNkRUoruuV4g3jSDIyXw9A', 't66qj4cbPYAYlfpqxchYvg'], ['-vPtYuV_dMPb0Bbzg9ECVw', 'KIAqCeQuhC2BQNI-p9AXLw'], ['EmurkIplcpOkh4hUnSs_ww', 'HKege4lBV85T-flscfoXgQ']]}]}}"]


In [11]:
userRDD = sc.textFile("data/yelp_academic_dataset_user.json")
targetUserID = "cRANsQ5E_sxeQ0Aoop3drg"
specificUserRDD = userRDD.map(json.loads).filter(lambda user: user['user_id'] == targetUserID)
specificUser = specificUserRDD.collect()
print("Search by user ID to pull all of a given flagged user's information for purpose of seeing what we are working with.\n")
print(specificUser[0])

Search by user ID to pull all of a given flagged user's information for purpose of seeing what we are working with.

{'user_id': 'cRANsQ5E_sxeQ0Aoop3drg', 'name': 'Amanda', 'review_count': 133, 'yelping_since': '2013-11-27 00:27:57', 'useful': 366, 'funny': 80, 'cool': 268, 'elite': '2017,2018,2019,20,20,2021', 'friends': 'Io4DrR1_vMLOVMdQsPATRA, LMIyGqwDMAVMWkCQb6sQ3Q, KlKgH4Te9EPb1uzC28Aw-w, ybN5igJTmnkFHXU4qseiDA, r12E0JbaVi4JCUgUa7b7fA, J2c6-nchh5ziQQuMOtm41A, ystMl1kYo0KMAnafP9hx6A, -SsCP5luPDPJyc1z0F4fHA, 3echCr8Y1XeW3Fq5Q_xn4w, vGDDsijFim4Zay8vlf6TDQ, kGd2F0sfE8WWAWty5uezNw, WHuuz1Xm8zkrUl8fAd31Eg, -ZTW3gcV_1DL1Eqx78DIAQ, du9gjqdmv6koj-kYpk3KQg, swobJ5rJuEGRKqHBf8EutA, 99HXweAV7ypgg0FNF0o8Jw, a5rkDUU_Ie2cDyktgtLyGQ, RaRJMKc-gXukTHFovDwiiw, EUkxMG8Bf6nGsrIXOCAQ4w, md5proR8D9ooUM_EHf7mdg, Vk_khMDkjXPkpffQXArFqA, 6oUE_bDkqfXap6zK5VWqIQ, nXgApfRajhkwCPPManEHPA, z4OYmpgfcaWMVIPGDKm_pg, XzYyKC84ccqgw0DccAgEsQ, q3_rELfkhP2JF8n78cBmRA, vwpYsmLp-g_VWbIt-vR3Qg, uM9OD-grlkb-MF3be1JQMw, T-E

In [12]:
topSuspiciousReviewers = sc.textFile("output/descAnalysis/topSuspiciousReviewers/part-00000").map(json.loads)
print("Sorted result of users with the highest count of suspected user reviews.\n\nFormat: User ID, # of flagged reviews, Name")
topSuspiciousReviewers.take(10)

Sorted result of users with the highest count of suspected user reviews.

Format: User ID, # of flagged reviews, Name


[['_BcWyKQL16ndpBdggh2kNA', [1315, 'Karen']],
 ['ET8n-r7glWYqZhuR6GcdNw', [1092, 'Michelle']],
 ['bJ5FtCtZX3ZZacz2_2PJjA', [475, 'Bill']],
 ['1HM81n6n4iPIFU5d2Lokhw', [461, 'Shannon']],
 ['CfX4sTIFFNaRchNswqhVfg', [399, 'Christopher']],
 ['E4BsVQnG5zetbwv2x8QIWg', [395, 'Charles']],
 ['pou3BbKsIozfH50rxmnMew', [367, 'Brett']],
 ['Ase_kJIYuT6yOsqqVPuWUA', [319, 'Teneha']],
 ['bYENop4BuQepBjM1-BI3fA', [317, 'Steven']],
 ['D8CF2H3DYtNJlzR0IITMHQ', [305, 'ThaDragonSourceRa ®']]]

In [13]:
topUsers = sc.textFile("output/descAnalysis/topUsers/part-00000")
print("\nFormat: Flagged Review Count, Total # of Friend Reviews, Total # of Flagged User's Reviews")
topUsers.take(10)


Format: Flagged Review Count, Total # of Friend Reviews, Total # of Flagged User's Reviews


['["_BcWyKQL16ndpBdggh2kNA", [[1315, 146544, 4274], "Karen"]]',
 '["ET8n-r7glWYqZhuR6GcdNw", [[1092, 381612, 2256], "Michelle"]]',
 '["bJ5FtCtZX3ZZacz2_2PJjA", [[475, 101598, 1512], "Bill"]]',
 '["1HM81n6n4iPIFU5d2Lokhw", [[461, 32947, 2317], "Shannon"]]',
 '["CfX4sTIFFNaRchNswqhVfg", [[399, 37223, 1405], "Christopher"]]',
 '["E4BsVQnG5zetbwv2x8QIWg", [[395, 34817, 726], "Charles"]]',
 '["pou3BbKsIozfH50rxmnMew", [[367, 204627, 2812], "Brett"]]',
 '["Ase_kJIYuT6yOsqqVPuWUA", [[319, 71234, 1134], "Teneha"]]',
 '["bYENop4BuQepBjM1-BI3fA", [[317, 82785, 1562], "Steven"]]',
 '["D8CF2H3DYtNJlzR0IITMHQ", [[305, 19989, 323], "ThaDragonSourceRa \\u00ae"]]']